# Step 12 — Offline Policy Evaluation: Tier 2 FCI-Guided Model

Evaluates the Tier 2 DDQN + SARSA models (step 11b) using:
- **Physician policy** — supervised π(a|s), used for importance weights in DR
- **Reward estimator** — R̂(s, a_agent), direct method component of DR
- **Doubly Robust (DR) off-policy evaluation** — estimates policy value
- **Fig 1** — drug distribution: physician vs DDQN, stratified by Shock_Index severity
- **Fig 2** — readmission rate vs action disagreement (Raghu 2017 Fig 2 equivalent)

**Key fix vs original step 12:** reward estimator is queried with the *agent's* action,
not the physician's action. The original used physician actions in the model-based term,
which biases DR toward the physician value when the policies disagree.

**Env model skipped:** The env model from the original step 12 is not used in the DR
formula and is not trained here.

**Before running:**
1. Set Runtime → T4 GPU
2. Ensure Drive has:
   - `MyDrive/CareAI/data/rl_dataset_tier2.parquet`
   - `MyDrive/CareAI/data/scaler_params_tier2.json`
   - `MyDrive/CareAI/models/tier2/ddqn/dqn_actions_test.pkl`
   - `MyDrive/CareAI/models/tier2/ddqn/dqn_q_test.pkl`
   - `MyDrive/CareAI/models/tier2/sarsa_phys/phys_actions_test.pkl`
   - `MyDrive/CareAI/models/tier2/sarsa_phys/phys_q_test.pkl`

In [ ]:
# ── Cell 1: Mount Drive + check GPU ──────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import torch
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: No GPU — training will be slow')

In [ ]:
# ── Cell 2: Install + imports ─────────────────────────────────────────────────
!pip install pyarrow --quiet

import json, os, pickle, time
import numpy as np
import pandas as pd
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

print('Imports OK')

In [ ]:
# ── Cell 3: Configuration ─────────────────────────────────────────────────────
DRIVE_ROOT   = '/content/drive/MyDrive/CareAI'
DATA_PATH    = os.path.join(DRIVE_ROOT, 'data', 'rl_dataset_tier2.parquet')
SCALER_PATH  = os.path.join(DRIVE_ROOT, 'data', 'scaler_params_tier2.json')
TIER2_DIR    = os.path.join(DRIVE_ROOT, 'models', 'tier2')
DDQN_DIR     = os.path.join(TIER2_DIR, 'ddqn')
SARSA_DIR    = os.path.join(TIER2_DIR, 'sarsa_phys')
EVAL_DIR     = os.path.join(TIER2_DIR, 'eval')
REPORT_DIR   = os.path.join(DRIVE_ROOT, 'reports', 'tier2')

os.makedirs(EVAL_DIR,   exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)

# Hyperparameters
N_STATE          = 5
N_ACTIONS        = 16
N_ACTION_BITS    = 4   # 4-bit encoding for tier2 (not 5-bit)
DRUG_NAMES       = ['vasopressor', 'ivfluid', 'antibiotic', 'diuretic']
PHYS_STEPS       = 35000
REWARD_STEPS     = 30000
BATCH_SIZE       = 512
LOG_EVERY        = 2000
GAMMA            = 0.99
VALUE_CLIP       = 40.0

print('Config OK')
print(f'  Eval dir:   {EVAL_DIR}')
print(f'  Report dir: {REPORT_DIR}')

In [ ]:
# ── Cell 4: Network architectures ────────────────────────────────────────────

class PhysicianPolicy(nn.Module):
    """Supervised pi(a|s): maps state -> action probabilities."""
    def __init__(self, n_input, n_actions=16, hidden=64):
        super().__init__()
        self.fc1 = nn.Linear(n_input, hidden)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.bn2 = nn.BatchNorm1d(hidden)
        self.out = nn.Linear(hidden, n_actions)

    def forward(self, x):
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        return self.out(x)

    def predict_proba(self, x):
        with torch.no_grad():
            return F.softmax(self.forward(x), dim=1)


class RewardEstimator(nn.Module):
    """R(s, a): maps (state + 4-bit action) -> scalar reward."""
    def __init__(self, n_state, n_action_bits=4, hidden=128):
        super().__init__()
        n_input = n_state + n_action_bits
        self.fc1 = nn.Linear(n_input, hidden)
        self.bn1 = nn.BatchNorm1d(hidden)
        self.fc2 = nn.Linear(hidden, hidden)
        self.bn2 = nn.BatchNorm1d(hidden)
        self.out = nn.Linear(hidden, 1)

    def forward(self, state, action_bin):
        x = torch.cat([state, action_bin], dim=1)
        x = F.relu(self.bn1(self.fc1(x)))
        x = F.relu(self.bn2(self.fc2(x)))
        return self.out(x).squeeze(-1)

print('Architectures defined')

In [ ]:
# ── Cell 5: Helper functions ──────────────────────────────────────────────────

def decode_actions_4bit(actions_int):
    """Decode integer actions 0-15 to (N, 4) binary array.
    Bit order: vasopressor=bit0, ivfluid=bit1, antibiotic=bit2, diuretic=bit3
    """
    arr = np.asarray(actions_int, dtype=np.int32)
    return ((arr[:, None] >> np.arange(4)) & 1).astype(np.float32)


def shock_severity(si_raw):
    """Map raw Shock_Index to low/medium/high severity labels.
    Shock_Index = HR / SysBP. Normal ~0.5-0.7. High (>1.0) = hemodynamic instability.
    """
    labels = np.full(len(si_raw), 'medium', dtype=object)
    labels[si_raw < 0.6] = 'low'
    labels[si_raw > 1.0] = 'high'
    return labels


# GPU-optimised physician policy training (no PER needed -- supervised learning)
def train_physician_policy(data, n_state, n_actions=16, hidden=64,
                           batch_size=512, num_steps=35000, reg_constant=0.1,
                           save_dir=None, device='cpu', log_every=2000):
    print(f'Physician policy: {num_steps} steps | n_state={n_state} | n_actions={n_actions} | device={device}')

    states_t  = torch.FloatTensor(data['states']).to(device)
    actions_t = torch.LongTensor(data['actions']).to(device)
    n = len(states_t)
    print(f'  {n:,} transitions on {device}')

    model     = PhysicianPolicy(n_state, n_actions, hidden).to(device)
    optimizer = optim.Adam(model.parameters(), weight_decay=reg_constant)
    criterion = nn.CrossEntropyLoss()

    losses_gpu = torch.zeros(num_steps, device=device)
    t0 = time.time()

    for step in range(num_steps):
        model.train()
        idx    = torch.randint(0, n, (batch_size,), device=device)
        logits = model(states_t[idx])
        loss   = criterion(logits, actions_t[idx])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses_gpu[step] = loss.detach()

        if (step + 1) % log_every == 0:
            elapsed   = time.time() - t0
            steps_sec = (step + 1) / elapsed
            eta       = (num_steps - step - 1) / steps_sec
            avg_loss  = losses_gpu[step+1-log_every:step+1].mean().item()
            acc       = (logits.argmax(dim=1) == actions_t[idx]).float().mean().item()
            print(f'  Phys {step+1}/{num_steps} ({100*(step+1)/num_steps:.0f}%) | '
                  f'loss={avg_loss:.4f} acc={acc:.3f} | {steps_sec:.1f} steps/s | ETA {eta:.0f}s')

    model.eval()
    print('Physician policy complete')

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(save_dir, 'physician_policy_model.pt'))

    return model


# GPU-optimised reward estimator training
def train_reward_estimator(data, n_state, n_action_bits=4, hidden=128,
                           batch_size=512, num_steps=30000, lr=1e-3,
                           save_dir=None, device='cpu', log_every=2000):
    print(f'Reward estimator: {num_steps} steps | n_state={n_state} | n_action_bits={n_action_bits} | device={device}')

    states_t      = torch.FloatTensor(data['states']).to(device)
    rewards_t     = torch.FloatTensor(data['rewards']).to(device)
    actions_bin_t = torch.FloatTensor(decode_actions_4bit(data['actions'])).to(device)
    n = len(states_t)
    print(f'  {n:,} transitions on {device}')

    model     = RewardEstimator(n_state, n_action_bits, hidden).to(device)
    optimizer = optim.Adam(model.parameters(), lr=lr)
    criterion = nn.MSELoss()

    losses_gpu = torch.zeros(num_steps, device=device)
    t0 = time.time()

    for step in range(num_steps):
        model.train()
        idx  = torch.randint(0, n, (batch_size,), device=device)
        pred = model(states_t[idx], actions_bin_t[idx])
        loss = criterion(pred, rewards_t[idx])

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        losses_gpu[step] = loss.detach()

        if (step + 1) % log_every == 0:
            elapsed   = time.time() - t0
            steps_sec = (step + 1) / elapsed
            eta       = (num_steps - step - 1) / steps_sec
            avg_loss  = losses_gpu[step+1-log_every:step+1].mean().item()
            print(f'  Rew {step+1}/{num_steps} ({100*(step+1)/num_steps:.0f}%) | '
                  f'loss={avg_loss:.4f} | {steps_sec:.1f} steps/s | ETA {eta:.0f}s')

    model.eval()
    print('Reward estimator complete')

    if save_dir:
        os.makedirs(save_dir, exist_ok=True)
        torch.save(model.state_dict(), os.path.join(save_dir, 'reward_estimator_model.pt'))

    return model


def predict_physician_probs(model, states_np, device='cpu', batch_size=4096):
    model.eval()
    states_t  = torch.FloatTensor(states_np).to(device)
    all_probs = []
    with torch.no_grad():
        for i in range(0, len(states_t), batch_size):
            all_probs.append(model.predict_proba(states_t[i:i+batch_size]).cpu().numpy())
    return np.concatenate(all_probs, axis=0)


def predict_agent_rewards(model, states_np, agent_actions, device='cpu', batch_size=4096):
    """R_hat(s, a_agent) -- reward estimator queried with the AGENT's action.
    This is the correct model-based term for DR: what reward would the agent's
    recommended action get, according to our reward model?
    """
    model.eval()
    states_t      = torch.FloatTensor(states_np).to(device)
    actions_bin_t = torch.FloatTensor(decode_actions_4bit(agent_actions)).to(device)
    all_rewards   = []
    with torch.no_grad():
        for i in range(0, len(states_t), batch_size):
            r = model(states_t[i:i+batch_size], actions_bin_t[i:i+batch_size])
            all_rewards.append(r.cpu().numpy())
    return np.concatenate(all_rewards, axis=0)


def doubly_robust_evaluation(data, agent_actions, agent_q_values,
                             physician_probs, agent_reward_estimates,
                             gamma=0.99, value_clip=40.0, label=''):
    """DR off-policy evaluation.

    Key fix: agent_reward_estimates = R_hat(s, a_agent), not R_hat(s, a_physician).
    When the agent disagrees with the physician, the model-based term now estimates
    the agent's expected reward, not the physician's.

    Returns array of per-trajectory DR values.
    """
    actions_taken = data['actions']
    rewards       = data['rewards']
    done          = data['done']
    n             = len(actions_taken)

    trajectories = []
    start = 0
    for i in range(1, n):
        if done[i-1] == 1.0:
            trajectories.append((start, i))
            start = i
    if start < n:
        trajectories.append((start, n))

    print(f'  {label}: {len(trajectories):,} trajectories')

    trajectory_values = []
    for traj_start, traj_end in trajectories:
        v_dr = 0.0
        for t in range(traj_end - 1, traj_start - 1, -1):
            a_phys  = int(actions_taken[t])
            a_agent = int(agent_actions[t])

            prob_phys = max(float(physician_probs[t, a_phys]), 1e-6)
            rho       = (1.0 / prob_phys) if (a_agent == a_phys) else 0.0

            q_agent = float(agent_q_values[t, a_agent])
            r_est   = float(agent_reward_estimates[t])  # R_hat(s, a_agent) -- fixed
            r_act   = float(rewards[t])

            v_dr = r_est + rho * (r_act + gamma * v_dr - q_agent)

        if abs(v_dr) <= value_clip:
            trajectory_values.append(v_dr)

    traj_vals = np.array(trajectory_values)
    print(f'  {label}: DR mean={traj_vals.mean():.4f} std={traj_vals.std():.4f} '
          f'valid={len(traj_vals)}/{len(trajectories)} ({100*len(traj_vals)/len(trajectories):.1f}%)')
    return traj_vals


print('Helper functions defined')

In [ ]:
# ── Cell 6: Load data ─────────────────────────────────────────────────────────
print(f'Loading {DATA_PATH} ...')
df = pd.read_parquet(DATA_PATH)
print(f'  {len(df):,} rows, {len(df.columns)} cols')
print(f'  Columns: {list(df.columns)}')

# Load scaler params for de-normalising Shock_Index (used for severity stratification)
with open(SCALER_PATH) as f:
    scaler = json.load(f)
SI_MEAN = scaler['Shock_Index']['mean']
SI_STD  = scaler['Shock_Index']['std']
print(f'  Shock_Index scaler: mean={SI_MEAN:.4f}, std={SI_STD:.4f}')

state_cols      = [c for c in df.columns if c.startswith('s_') and not c.startswith('s_next_')]
next_state_cols = [c.replace('s_', 's_next_', 1) for c in state_cols]
n_state         = len(state_cols)
print(f'  State features ({n_state}): {state_cols}')
print(f'  Actions: {df["a"].nunique()} unique (expect {N_ACTIONS})')
print(f'  Reward: mean={df["r"].mean():.3f}, std={df["r"].std():.3f}')

df_train = df[df['split'] == 'train'].reset_index(drop=True)
df_val   = df[df['split'] == 'val'].reset_index(drop=True)
df_test  = df[df['split'] == 'test'].reset_index(drop=True)
print(f'  Splits: train={len(df_train):,}, val={len(df_val):,}, test={len(df_test):,}')
del df

def to_rl_arrays(df):
    return {
        'states':      df[state_cols].values.astype(np.float32),
        'next_states': df[next_state_cols].values.astype(np.float32),
        'actions':     df['a'].values.astype(int),
        'rewards':     df['r'].values.astype(np.float32),
        'done':        df['done'].values.astype(np.float32),
    }

train_data = to_rl_arrays(df_train)
test_data  = to_rl_arrays(df_test)

# Shock_Index raw values for severity stratification in test set
shock_raw_test = df_test['s_Shock_Index'].values * SI_STD + SI_MEAN
sev_labels_test = shock_severity(shock_raw_test)
readmit_test    = df_test['readmit_30d'].values.astype(float)

for sev in ['low', 'medium', 'high']:
    n = (sev_labels_test == sev).sum()
    print(f'  Severity {sev}: {n:,} timesteps ({100*n/len(sev_labels_test):.1f}%)')

print('Data ready.')

In [ ]:
# ── Cell 7: SMOKE TEST ────────────────────────────────────────────────────────
# 200 steps each. Checks shapes, finiteness, DR runs without error.
# Must pass before proceeding to full training.

SMOKE_DIR   = '/tmp/smoke_eval_tier2'
SMOKE_STEPS = 200

print('=' * 60)
print('SMOKE TEST')
print('=' * 60)

sm_phys  = train_physician_policy(train_data, N_STATE, N_ACTIONS,
                                   num_steps=SMOKE_STEPS, device=device, log_every=100)
sm_rew   = train_reward_estimator(train_data, N_STATE, N_ACTION_BITS,
                                   num_steps=SMOKE_STEPS, device=device, log_every=100)

# Check shapes
sm_probs  = predict_physician_probs(sm_phys, test_data['states'], device=device)
sm_rews   = predict_agent_rewards(sm_rew, test_data['states'],
                                   agent_actions=test_data['actions'], device=device)

errors = []
if sm_probs.shape != (len(test_data['states']), N_ACTIONS):
    errors.append(f'Physician probs shape: {sm_probs.shape}')
if not np.isfinite(sm_probs).all():
    errors.append('Physician probs contain NaN/Inf')
if sm_rews.shape != (len(test_data['states']),):
    errors.append(f'Agent reward estimates shape: {sm_rews.shape}')
if not np.isfinite(sm_rews).all():
    errors.append('Agent reward estimates contain NaN/Inf')
if not np.allclose(sm_probs.sum(axis=1), 1.0, atol=1e-4):
    errors.append('Physician probs do not sum to 1')

# Quick DR smoke (uses dummy Q-values)
dummy_q = np.zeros((len(test_data['states']), N_ACTIONS), dtype=np.float32)
dr_smoke = doubly_robust_evaluation(test_data, test_data['actions'], dummy_q,
                                     sm_probs, sm_rews, label='smoke')
if len(dr_smoke) == 0:
    errors.append('DR returned 0 valid trajectories')

print()
if errors:
    for e in errors: print(f'  FAIL: {e}')
    raise RuntimeError('Smoke test FAILED')
else:
    print(f'  Physician probs shape:  {sm_probs.shape}  -- OK')
    print(f'  Probs sum to 1:         {np.allclose(sm_probs.sum(1),1,atol=1e-4)}  -- OK')
    print(f'  Agent rewards shape:    {sm_rews.shape}  -- OK')
    print(f'  DR valid trajectories:  {len(dr_smoke)}  -- OK')
    print()
    print('SMOKE TEST PASSED')

In [ ]:
# ── Cell 8: Train physician policy ────────────────────────────────────────────
phys_model = train_physician_policy(
    train_data, N_STATE, N_ACTIONS,
    hidden=64, batch_size=BATCH_SIZE, num_steps=PHYS_STEPS,
    save_dir=os.path.join(EVAL_DIR, 'physician_policy'),
    device=device, log_every=LOG_EVERY,
)

# Predict for test set
test_phys_probs = predict_physician_probs(phys_model, test_data['states'], device=device)
print(f'Test physician probs: shape={test_phys_probs.shape}, '
      f'mean_entropy={(-test_phys_probs * np.log(test_phys_probs + 1e-9)).sum(1).mean():.3f}')

In [ ]:
# ── Cell 9: Train reward estimator ────────────────────────────────────────────
rew_model = train_reward_estimator(
    train_data, N_STATE, N_ACTION_BITS,
    hidden=128, batch_size=BATCH_SIZE, num_steps=REWARD_STEPS,
    save_dir=os.path.join(EVAL_DIR, 'reward_estimator'),
    device=device, log_every=LOG_EVERY,
)

# Quick validation: reward prediction on test set with physician actions
test_phys_rew_est = predict_agent_rewards(rew_model, test_data['states'],
                                          test_data['actions'], device=device)
corr = np.corrcoef(test_phys_rew_est, test_data['rewards'])[0,1]
print(f'Reward estimator (phys actions) vs actual rewards: r={corr:.4f}')
print(f'  Predicted: mean={test_phys_rew_est.mean():.3f}, std={test_phys_rew_est.std():.3f}')
print(f'  Actual:    mean={test_data["rewards"].mean():.3f}, std={test_data["rewards"].std():.3f}')

In [ ]:
# ── Cell 10: Load DDQN and SARSA model predictions ───────────────────────────

def load_pkl(path):
    with open(path, 'rb') as f:
        return pickle.load(f)

ddqn_actions_test = load_pkl(os.path.join(DDQN_DIR,  'dqn_actions_test.pkl'))
ddqn_q_test       = load_pkl(os.path.join(DDQN_DIR,  'dqn_q_test.pkl'))
sarsa_actions_test = load_pkl(os.path.join(SARSA_DIR, 'phys_actions_test.pkl'))
sarsa_q_test       = load_pkl(os.path.join(SARSA_DIR, 'phys_q_test.pkl'))

print(f'DDQN actions test:  {ddqn_actions_test.shape}, unique={len(np.unique(ddqn_actions_test))}')
print(f'DDQN Q test:        {ddqn_q_test.shape}')
print(f'SARSA actions test: {sarsa_actions_test.shape}, unique={len(np.unique(sarsa_actions_test))}')
print(f'SARSA Q test:       {sarsa_q_test.shape}')
assert len(ddqn_actions_test) == len(test_data['states']), \
    f'DDQN action length {len(ddqn_actions_test)} != test set {len(test_data["states"])}'
print('Shapes OK')

# Pre-compute R_hat(s, a_agent) for both policies
print('\nPre-computing R_hat(s, a_agent) for DDQN...')
ddqn_agent_rew_est = predict_agent_rewards(rew_model, test_data['states'],
                                           ddqn_actions_test, device=device)
print('Pre-computing R_hat(s, a_agent) for SARSA physician...')
sarsa_agent_rew_est = predict_agent_rewards(rew_model, test_data['states'],
                                            sarsa_actions_test, device=device)
print('Done.')

In [ ]:
# ── Cell 11: Doubly Robust evaluation ────────────────────────────────────────
print('=== Doubly Robust Off-Policy Evaluation ===')
print(f'gamma={GAMMA}, value_clip={VALUE_CLIP}')
print()

# DDQN policy
ddqn_dr = doubly_robust_evaluation(
    test_data, ddqn_actions_test, ddqn_q_test,
    test_phys_probs, ddqn_agent_rew_est,
    gamma=GAMMA, value_clip=VALUE_CLIP, label='DDQN'
)

# SARSA physician baseline
sarsa_dr = doubly_robust_evaluation(
    test_data, sarsa_actions_test, sarsa_q_test,
    test_phys_probs, sarsa_agent_rew_est,
    gamma=GAMMA, value_clip=VALUE_CLIP, label='SARSA-physician'
)

# Disagreement rates
ddqn_disagree  = (ddqn_actions_test  != test_data['actions']).mean()
sarsa_disagree = (sarsa_actions_test != test_data['actions']).mean()
print(f'\nAction disagreement: DDQN={ddqn_disagree:.1%}  SARSA={sarsa_disagree:.1%}')

dr_results = {
    'ddqn':  {'dr_mean': float(ddqn_dr.mean()),  'dr_std': float(ddqn_dr.std()),
              'n_valid': len(ddqn_dr),  'disagreement_pct': float(ddqn_disagree)},
    'sarsa': {'dr_mean': float(sarsa_dr.mean()), 'dr_std': float(sarsa_dr.std()),
              'n_valid': len(sarsa_dr), 'disagreement_pct': float(sarsa_disagree)},
}

In [ ]:
# ── Cell 12: Fig 1 — Drug distribution by severity ───────────────────────────
SEV_BINS   = ['low', 'medium', 'high']
SEV_COLORS = {'low': '#2196F3', 'medium': '#FF9800', 'high': '#F44336'}

phys_bin_test = decode_actions_4bit(test_data['actions'])
ddqn_bin_test = decode_actions_4bit(ddqn_actions_test)

fig, axes = plt.subplots(1, 3, figsize=(16, 5), sharey=True)
fig.suptitle('Fig 1 — Drug Usage Frequency: Physician vs DDQN by Shock_Index Severity',
             fontsize=13, fontweight='bold')

x     = np.arange(len(DRUG_NAMES))
width = 0.35
fig1_stats = {}

for ax, sev in zip(axes, SEV_BINS):
    mask       = sev_labels_test == sev
    n_sev      = mask.sum()
    phys_rates = phys_bin_test[mask].mean(axis=0) * 100
    ddqn_rates = ddqn_bin_test[mask].mean(axis=0) * 100

    ax.bar(x - width/2, phys_rates, width, label='Physician', color='#455A64', alpha=0.85)
    ax.bar(x + width/2, ddqn_rates, width, label='DDQN',      color=SEV_COLORS[sev], alpha=0.85)
    ax.set_title(f'Shock_Index severity: {sev.upper()}\n(n={n_sev:,} timesteps)', fontsize=11)
    ax.set_xticks(x)
    ax.set_xticklabels(DRUG_NAMES, rotation=20, ha='right')
    ax.set_ylabel('% timesteps drug active' if sev == 'low' else '')
    ax.set_ylim(0, 100)
    ax.legend(fontsize=9)
    ax.grid(axis='y', alpha=0.3)

    fig1_stats[sev] = {
        'n_timesteps': int(n_sev),
        'physician': {d: float(r) for d, r in zip(DRUG_NAMES, phys_rates)},
        'ddqn':      {d: float(r) for d, r in zip(DRUG_NAMES, ddqn_rates)},
    }

plt.tight_layout()
fig1_path = os.path.join(REPORT_DIR, 'fig1_drug_distribution.png')
plt.savefig(fig1_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Fig 1 saved: {fig1_path}')

In [ ]:
# ── Cell 13: Fig 2 — Readmission rate vs action disagreement ─────────────────
# Hamming distance (0-4): number of drugs where DDQN and physician disagree.
# If the model is valid: readmission should be worst where disagreement is highest.
# This is the Raghu 2017 Fig 2 equivalent.

hamming_test = (phys_bin_test != ddqn_bin_test).sum(axis=1)  # 0-4 per timestep

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
fig.suptitle(
    'Fig 2 — Readmission Rate vs DDQN-Physician Disagreement by Shock_Index Severity\n'
    '(Raghu 2017 Fig 2 equivalent: readmission should be worst where disagreement is highest)',
    fontsize=11, fontweight='bold'
)

fig2_stats = {}

for ax, sev in zip(axes, SEV_BINS):
    mask   = sev_labels_test == sev
    h_sev  = hamming_test[mask]
    r_sev  = readmit_test[mask]

    bin_rates  = []
    bin_ns     = []

    for d in range(5):  # 0-4 drugs different
        bin_mask = h_sev == d
        n_bin    = bin_mask.sum()
        rate     = r_sev[bin_mask].mean() * 100 if n_bin > 0 else np.nan
        bin_rates.append(rate)
        bin_ns.append(int(n_bin))

    bin_rates = np.array(bin_rates)
    valid     = ~np.isnan(bin_rates)

    ax.bar(np.arange(5)[valid], bin_rates[valid],
           color=SEV_COLORS[sev], alpha=0.8, edgecolor='black', linewidth=0.5)

    for d in range(5):
        if valid[d] and bin_ns[d] > 0:
            ax.text(d, bin_rates[d] + 0.3, f'n={bin_ns[d]:,}',
                    ha='center', va='bottom', fontsize=7, rotation=45)

    ax.axvline(x=0, color='black', linestyle='--', linewidth=1.2, alpha=0.6,
               label='Physician agrees with DDQN')
    ax.set_title(f'Shock_Index severity: {sev.upper()}', fontsize=11)
    ax.set_xlabel('Drugs where DDQN and physician disagree')
    ax.set_ylabel('Readmission rate (%)' if sev == 'low' else '')
    ax.set_xticks(range(5))
    ax.set_xticklabels([f'{d} drug{"s" if d!=1 else ""}' for d in range(5)],
                       rotation=20, ha='right', fontsize=9)
    ax.legend(fontsize=8)
    ax.grid(axis='y', alpha=0.3)

    fig2_stats[sev] = {
        'hamming_bins': {
            str(d): {'readmission_rate_pct': float(bin_rates[d]) if valid[d] else None,
                     'n_timesteps': bin_ns[d]}
            for d in range(5)
        }
    }

plt.tight_layout()
fig2_path = os.path.join(REPORT_DIR, 'fig2_readmission_vs_disagreement.png')
plt.savefig(fig2_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Fig 2 saved: {fig2_path}')

In [ ]:
# ── Cell 14: Save all results ─────────────────────────────────────────────────
all_results = {
    'dr_evaluation': dr_results,
    'fig1_drug_distribution': fig1_stats,
    'fig2_readmission_vs_disagreement': fig2_stats,
    'reward_estimator_correlation': float(corr),
    'action_disagreement': {
        'ddqn_vs_physician_pct':  float(ddqn_disagree  * 100),
        'sarsa_vs_physician_pct': float(sarsa_disagree * 100),
    },
}

results_path = os.path.join(REPORT_DIR, 'evaluation_results.json')
with open(results_path, 'w') as f:
    json.dump(all_results, f, indent=2)

print('=== EVALUATION SUMMARY ===')
print(f'DR (DDQN):             {dr_results["ddqn"]["dr_mean"]:+.4f} +/- {dr_results["ddqn"]["dr_std"]:.4f}')
print(f'DR (SARSA physician):  {dr_results["sarsa"]["dr_mean"]:+.4f} +/- {dr_results["sarsa"]["dr_std"]:.4f}')
print(f'Reward estimator r:    {corr:.4f}')
print(f'DDQN disagrees with physician: {ddqn_disagree:.1%}')
print()
print(f'Results saved: {results_path}')
print(f'Fig 1:         {fig1_path}')
print(f'Fig 2:         {fig2_path}')
print()
print('Download reports/tier2/ from Drive to CareAI/reports/icu_readmit/tier2/')

---
## Part 2: Tier 2 Discharge Analysis (Step 11c)

The discharge analysis does not require training new models — the discharge destination
is observable and the physician's choice is in the data. We compare:
- Physician discharge distribution vs DDQN/SARSA recommendations
- Actual 30-day readmission by discharge destination

**Before running:** Ensure Drive has:
- `MyDrive/CareAI/data/rl_dataset_tier2_discharge.parquet`
- `MyDrive/CareAI/models/tier2_discharge/ddqn/discharge_actions.pkl`
- `MyDrive/CareAI/models/tier2_discharge/sarsa_phys/phys_discharge_actions.pkl`

In [ ]:
# ── Cell 15: Load discharge parquet ──────────────────────────────────────────
DISCHARGE_PATH  = os.path.join(DRIVE_ROOT, 'data', 'rl_dataset_tier2_discharge.parquet')
TIER2D_DIR      = os.path.join(DRIVE_ROOT, 'models', 'tier2_discharge')
DISCHARGE_REPORT = os.path.join(DRIVE_ROOT, 'reports', 'tier2_discharge')
os.makedirs(DISCHARGE_REPORT, exist_ok=True)

print(f'Loading {DISCHARGE_PATH} ...')
df_disc = pd.read_parquet(DISCHARGE_PATH)
print(f'  {len(df_disc):,} rows, {len(df_disc.columns)} cols')
print(f'  Columns: {list(df_disc.columns)}')
print(f'  Unique actions: {sorted(df_disc["a"].unique().tolist())}')

# Identify phase-0 (drug) and phase-1 (discharge) rows
# The discharge rows are the terminal rows with discharge actions (values 0, 1, 2)
# Drug rows have actions 0-15; discharge rows have actions 0, 1, 2
# Use the 'phase' column if it exists, otherwise infer from action range + done flag
if 'phase' in df_disc.columns:
    phase0 = df_disc[df_disc['phase'] == 0]
    phase1 = df_disc[df_disc['phase'] == 1]
    print(f'  Phase 0 (drug):      {len(phase0):,} rows')
    print(f'  Phase 1 (discharge): {len(phase1):,} rows')
else:
    # Infer: discharge rows are terminal (done=1) with low action values
    # This depends on step_10d implementation -- print structure to inspect
    print('  No phase column found -- inspecting structure:')
    print(f'  done=0: {(df_disc["done"]==0).sum():,}  done=1: {(df_disc["done"]==1).sum():,}')
    print(f'  action value counts (top 10):')
    print(df_disc['a'].value_counts().head(10))

In [ ]:
# ── Cell 16: Discharge distribution analysis ──────────────────────────────────
# Load model discharge action recommendations
# NOTE: file names depend on step_11c output structure -- adjust if needed

DISC_CATS     = [0, 1, 2]
DISC_LABELS   = ['Home', 'Home+Services', 'Institutional']
DISC_COLORS   = ['#4CAF50', '#FF9800', '#F44336']

try:
    ddqn_disc_actions  = load_pkl(os.path.join(TIER2D_DIR, 'ddqn',  'discharge_actions.pkl'))
    sarsa_disc_actions = load_pkl(os.path.join(TIER2D_DIR, 'sarsa_phys', 'phys_discharge_actions.pkl'))
    print(f'DDQN discharge actions:  {len(ddqn_disc_actions):,} unique={np.unique(ddqn_disc_actions).tolist()}')
    print(f'SARSA discharge actions: {len(sarsa_disc_actions):,} unique={np.unique(sarsa_disc_actions).tolist()}')
except FileNotFoundError as e:
    print(f'File not found: {e}')
    print('Adjust file paths in TIER2D_DIR to match step_11c output names')
    raise

# Filter to phase-1 (discharge) rows in test split
# Adjust the filter condition based on Cell 15 findings
if 'phase' in df_disc.columns:
    disc_test = df_disc[(df_disc['phase'] == 1) & (df_disc['split'] == 'test')].reset_index(drop=True)
else:
    # Fallback: terminal rows in test split
    disc_test = df_disc[(df_disc['done'] == 1) & (df_disc['split'] == 'test')].reset_index(drop=True)

print(f'Discharge test rows: {len(disc_test):,}')

phys_disc  = disc_test['a'].values
readmit_disc = disc_test['readmit_30d'].values

# Align model arrays to discharge test rows
# If model arrays span all test rows, slice to discharge rows
# (depends on how step_11c saved them -- may need adjustment)
n_disc_test = len(disc_test)
ddqn_disc_test  = ddqn_disc_actions[-n_disc_test:]   # last N rows = discharge rows
sarsa_disc_test = sarsa_disc_actions[-n_disc_test:]

# Distribution comparison
fig, axes = plt.subplots(1, 3, figsize=(14, 5))
fig.suptitle('Fig 3 — Discharge Destination: Physician vs Model Recommendations\n'
             '(Phase 1 terminal action — 30-day readmission rates by destination)',
             fontsize=11, fontweight='bold')

policies = [
    ('Physician\n(observed)', phys_disc),
    ('DDQN\npolicy',         ddqn_disc_test),
    ('SARSA\nphysician',     sarsa_disc_test),
]

disc_stats = {}

for ax, (label, actions) in zip(axes, policies):
    counts = [100 * (actions == c).mean() for c in DISC_CATS]
    bars   = ax.bar(DISC_LABELS, counts, color=DISC_COLORS, alpha=0.85, edgecolor='black')

    # Readmission rate overlay on actual physician discharge
    if label.startswith('Physician'):
        ax2 = ax.twinx()
        readmit_by_dest = [100 * readmit_disc[actions == c].mean() if (actions == c).sum() > 0
                           else np.nan for c in DISC_CATS]
        ax2.plot(DISC_LABELS, readmit_by_dest, 'k--o', linewidth=2, markersize=6,
                 label='Readmit rate %')
        ax2.set_ylabel('Readmission rate (%)', color='black')
        ax2.legend(loc='upper right', fontsize=8)
        disc_stats['readmit_by_actual_destination'] = {
            DISC_LABELS[i]: float(readmit_by_dest[i]) if not np.isnan(readmit_by_dest[i]) else None
            for i in range(3)
        }

    for bar, pct in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                f'{pct:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

    ax.set_title(label, fontsize=12)
    ax.set_ylabel('% of discharge decisions' if label.startswith('Physician') else '')
    ax.set_ylim(0, 75)
    ax.grid(axis='y', alpha=0.3)

    disc_stats[label.replace('\n','_').strip()] = {DISC_LABELS[i]: float(counts[i]) for i in range(3)}

plt.tight_layout()
fig3_path = os.path.join(DISCHARGE_REPORT, 'fig3_discharge_distribution.png')
plt.savefig(fig3_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Fig 3 saved: {fig3_path}')

# Disagreement rate
disc_disagree = (ddqn_disc_test != phys_disc).mean()
print(f'\nDDQN vs physician discharge disagreement: {disc_disagree:.1%}')

disc_results_path = os.path.join(DISCHARGE_REPORT, 'discharge_analysis.json')
with open(disc_results_path, 'w') as f:
    json.dump(disc_stats, f, indent=2)
print(f'Discharge stats saved: {disc_results_path}')